<a href="https://colab.research.google.com/github/JSJeong-me/MCP/blob/main/02%20Agent%20Tool/2_mcp_module2_host_client_server_intro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MCP Host / Client / Server 입문 실습

이 노트북은 MCP의 핵심 구조인 **Host / Client / Server** 관계와,
**JSON-RPC 2.0 기반 메시지 흐름**을 가장 단순한 형태로 이해하기 위한 입문 실습입니다.

실습 목표:
- Host, Client, Server의 역할 구분
- `initialize -> initialized -> tools/list -> tools/call` 흐름 이해
- JSON-RPC 메시지를 눈으로 확인


## 실습 실행 순서

아래 순서대로 셀을 실행하세요.

1. **기본 import 실행**
2. **JSON-RPC 메시지 헬퍼 정의**
3. **Demo MCP Server 정의**
4. **Demo MCP Client 정의**
5. **Host App 정의**
6. **전체 데모 실행**

권장 실행 방법:
- 메뉴에서 **런타임 → 모두 실행**
- 또는 위에서 아래로 한 셀씩 실행


In [1]:
# 1. 기본 import
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional
import json
import uuid

print('환경 준비 완료')

환경 준비 완료


## 1) JSON-RPC 메시지 헬퍼

MCP는 JSON-RPC 2.0 기반으로 요청, 응답, 알림을 주고받습니다.
이 셀은 그 메시지를 만들기 위한 도우미 함수입니다.

In [2]:
def rpc_request(msg_id: int, method: str, params: Dict[str, Any]) -> Dict[str, Any]:
    return {
        'jsonrpc': '2.0',
        'id': msg_id,
        'method': method,
        'params': params,
    }

def rpc_response(msg_id: int, result: Dict[str, Any]) -> Dict[str, Any]:
    return {
        'jsonrpc': '2.0',
        'id': msg_id,
        'result': result,
    }

def rpc_error(msg_id: int, code: int, message: str) -> Dict[str, Any]:
    return {
        'jsonrpc': '2.0',
        'id': msg_id,
        'error': {
            'code': code,
            'message': message,
        },
    }

def rpc_notification(method: str, params: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
    msg = {
        'jsonrpc': '2.0',
        'method': method,
    }
    if params is not None:
        msg['params'] = params
    return msg

def pretty_print(title: str, payload: Dict[str, Any]) -> None:
    print('\n' + '=' * 80)
    print(title)
    print('=' * 80)
    print(json.dumps(payload, ensure_ascii=False, indent=2))

print('JSON-RPC 헬퍼 정의 완료')

JSON-RPC 헬퍼 정의 완료


## 2) Demo MCP Server

Server는 기능 제공자입니다.
이 예제에서는 `add_numbers`라는 하나의 tool만 제공합니다.
또한 초기화 전 일반 작업을 거부합니다.

In [3]:
@dataclass
class DemoMCPServer:
    name: str = 'demo-math-server'
    version: str = '1.0.0'
    supported_protocol_version: str = '2025-06-18'
    initialized_sessions: Dict[str, bool] = field(default_factory=dict)

    def handle_message(self, session_id: str, message: Dict[str, Any]) -> Dict[str, Any]:
        method = message.get('method')

        if method == 'initialize':
            return self._handle_initialize(message)

        if method == 'notifications/initialized':
            self.initialized_sessions[session_id] = True
            return {'notification_ack': True}

        if not self.initialized_sessions.get(session_id, False):
            return rpc_error(message.get('id', 0), -32002, 'Session not initialized')

        if method == 'tools/list':
            return self._handle_tools_list(message)

        if method == 'tools/call':
            return self._handle_tools_call(message)

        return rpc_error(message.get('id', 0), -32601, f'Method not found: {method}')

    def _handle_initialize(self, message: Dict[str, Any]) -> Dict[str, Any]:
        result = {
            'protocolVersion': self.supported_protocol_version,
            'capabilities': {
                'tools': {}
            },
            'serverInfo': {
                'name': self.name,
                'version': self.version,
            }
        }
        return rpc_response(message['id'], result)

    def _handle_tools_list(self, message: Dict[str, Any]) -> Dict[str, Any]:
        result = {
            'tools': [
                {
                    'name': 'add_numbers',
                    'description': '두 정수를 더합니다.',
                    'inputSchema': {
                        'type': 'object',
                        'properties': {
                            'a': {'type': 'integer'},
                            'b': {'type': 'integer'},
                        },
                        'required': ['a', 'b'],
                    },
                }
            ]
        }
        return rpc_response(message['id'], result)

    def _handle_tools_call(self, message: Dict[str, Any]) -> Dict[str, Any]:
        params = message.get('params', {})
        tool_name = params.get('name')
        arguments = params.get('arguments', {})

        if tool_name != 'add_numbers':
            return rpc_error(message['id'], -32602, f'Unknown tool: {tool_name}')

        a = arguments.get('a')
        b = arguments.get('b')

        if not isinstance(a, int) or not isinstance(b, int):
            return rpc_error(message['id'], -32602, 'a and b must be integers')

        value = a + b
        result = {
            'content': [
                {
                    'type': 'text',
                    'text': f'덧셈 결과는 {value} 입니다.'
                }
            ],
            'structuredContent': {
                'a': a,
                'b': b,
                'sum': value,
            },
            'isError': False
        }
        return rpc_response(message['id'], result)

print('DemoMCPServer 정의 완료')

DemoMCPServer 정의 완료


## 3) Demo MCP Client

Client는 Host 안에서 Server와의 연결과 세션을 관리합니다.

In [4]:
@dataclass
class DemoMCPClient:
    server: DemoMCPServer
    client_name: str = 'demo-client'
    client_version: str = '0.1.0'
    protocol_version: str = '2025-06-18'
    session_id: str = field(default_factory=lambda: str(uuid.uuid4()))
    message_id: int = 0
    connected: bool = False
    initialized: bool = False

    def _next_id(self) -> int:
        self.message_id += 1
        return self.message_id

    def send_request(self, method: str, params: Dict[str, Any]) -> Dict[str, Any]:
        msg = rpc_request(self._next_id(), method, params)
        pretty_print('CLIENT -> SERVER (REQUEST)', msg)
        resp = self.server.handle_message(self.session_id, msg)
        pretty_print('SERVER -> CLIENT (RESPONSE)', resp)
        return resp

    def send_notification(self, method: str, params: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
        msg = rpc_notification(method, params)
        pretty_print('CLIENT -> SERVER (NOTIFICATION)', msg)
        resp = self.server.handle_message(self.session_id, msg)
        pretty_print('SERVER INTERNAL ACK', resp)
        return resp

    def connect(self) -> None:
        resp = self.send_request(
            'initialize',
            {
                'protocolVersion': self.protocol_version,
                'capabilities': {
                    'roots': {},
                    'sampling': {}
                },
                'clientInfo': {
                    'name': self.client_name,
                    'version': self.client_version,
                }
            }
        )

        if 'error' in resp:
            raise RuntimeError(f"Initialize failed: {resp['error']}")

        negotiated = resp['result']['protocolVersion']
        self.connected = True
        self.initialized = True
        print(f'\n[CLIENT] Negotiated protocol version: {negotiated}')
        self.send_notification('notifications/initialized')

    def list_tools(self) -> Dict[str, Any]:
        if not self.initialized:
            raise RuntimeError('Client is not initialized')
        return self.send_request('tools/list', {})

    def call_tool(self, name: str, arguments: Dict[str, Any]) -> Dict[str, Any]:
        if not self.initialized:
            raise RuntimeError('Client is not initialized')
        return self.send_request(
            'tools/call',
            {
                'name': name,
                'arguments': arguments,
            }
        )

print('DemoMCPClient 정의 완료')

DemoMCPClient 정의 완료


## 4) Host App

Host는 실제 사용자와 만나는 응용 프로그램입니다.

In [5]:
@dataclass
class HostApp:
    name: str = 'demo-host-app'
    clients: List[DemoMCPClient] = field(default_factory=list)

    def attach_server(self, server: DemoMCPServer) -> DemoMCPClient:
        client = DemoMCPClient(server=server)
        self.clients.append(client)
        print(f'[HOST] New client attached. total_clients={len(self.clients)}')
        return client

    def run_demo(self) -> None:
        print(f'[HOST] Starting {self.name}')

        server = DemoMCPServer()
        client = self.attach_server(server)

        print('\n[HOST] Step 1. Initialize session')
        client.connect()

        print('\n[HOST] Step 2. Ask what tools the server provides')
        tools_resp = client.list_tools()
        tool_names = [tool['name'] for tool in tools_resp['result']['tools']]
        print(f'\n[HOST] Available tools: {tool_names}')

        print('\n[HOST] Step 3. Call add_numbers tool')
        call_resp = client.call_tool('add_numbers', {'a': 7, 'b': 8})

        text = call_resp['result']['content'][0]['text']
        structured = call_resp['result']['structuredContent']

        print('\n[HOST] Final result shown to user')
        print('text:', text)
        print('structuredContent:', structured)

print('HostApp 정의 완료')

HostApp 정의 완료


## 5) 전체 데모 실행

이 셀을 실행하면 전체 MCP 입문 흐름이 한 번에 보입니다.

In [6]:
app = HostApp()
app.run_demo()

[HOST] Starting demo-host-app
[HOST] New client attached. total_clients=1

[HOST] Step 1. Initialize session

CLIENT -> SERVER (REQUEST)
{
  "jsonrpc": "2.0",
  "id": 1,
  "method": "initialize",
  "params": {
    "protocolVersion": "2025-06-18",
    "capabilities": {
      "roots": {},
      "sampling": {}
    },
    "clientInfo": {
      "name": "demo-client",
      "version": "0.1.0"
    }
  }
}

SERVER -> CLIENT (RESPONSE)
{
  "jsonrpc": "2.0",
  "id": 1,
  "result": {
    "protocolVersion": "2025-06-18",
    "capabilities": {
      "tools": {}
    },
    "serverInfo": {
      "name": "demo-math-server",
      "version": "1.0.0"
    }
  }
}

[CLIENT] Negotiated protocol version: 2025-06-18

CLIENT -> SERVER (NOTIFICATION)
{
  "jsonrpc": "2.0",
  "method": "notifications/initialized"
}

SERVER INTERNAL ACK
{
  "notification_ack": true
}

[HOST] Step 2. Ask what tools the server provides

CLIENT -> SERVER (REQUEST)
{
  "jsonrpc": "2.0",
  "id": 2,
  "method": "tools/list",
  "params"

## 실습 후 정리 질문

1. Host와 Client는 왜 분리되어 있을까요?
2. 왜 `initialize` 전에 `tools/call`을 하면 안 될까요?
3. `tools/list`와 `tools/call`의 차이는 무엇일까요?
4. Server를 2개로 늘리면 Host 쪽 구조는 어떻게 바뀔까요?
